# YOLO Ensemble Zoom-ROI Video Detection

This notebook runs the full ensemble model on `0703.mp4`, creates a zoom-in turbine inspection layer, saves detection video chunks every 10%, and clears old logs/cache after each chunk to reduce memory pressure.

Put this notebook, `0703.mp4`, and all `.pt` model files in the same folder, or set `MODEL_DIR` and `VIDEO_FILE` in the settings cell.


## 1. Install required packages

Run this cell once if your environment does not already have the required packages.  
After installation, restart the Jupyter kernel before running the detection cells.


In [1]:
import sys

!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install opencv-python-headless ultralytics pandas numpy matplotlib pyyaml tqdm


## 2. Check current notebook folder

Use this to confirm that the notebook can see your video and model files.


In [2]:
from pathlib import Path

print("Current folder:", Path.cwd())
print("Video exists:", (Path.cwd() / "0703.mp4").exists())

for name in [
    "yolov8s.pt",
    "yolo11s_norm750.pt",
    "yolo12s_norm750.pt",
    "yolov8s_norm1500.pt",
    "yolov10s_norm750.pt",
    "rtdetr-l.pt",
]:
    print(f"{name}:", (Path.cwd() / name).exists())


Current folder: /home/jovyan/work/AIFFEL_QUEST_ENG/Main_Quest/MQ02
Video exists: True
yolov8s.pt: True
yolo11s_norm750.pt: True
yolo12s_norm750.pt: True
yolov8s_norm1500.pt: True
yolov10s_norm750.pt: True
rtdetr-l.pt: True


## 3. Detection code

Run the following code cells once. The final execution cell starts the video detection.


In [3]:
from pathlib import Path
from collections import Counter, defaultdict
import time
import gc

import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO

try:
    import torch
except Exception:
    torch = None


### 0. USER SETTINGS


In [4]:
# ============================================================
# 0. USER SETTINGS
# ============================================================

# Folder that contains the ensemble .pt files.
# Keep this path as your original project/model folder.
MODEL_DIR = Path.cwd()

# The video file you want to detect.
# The resolver below searches:
# 1) the same folder as this .py file
# 2) the current working folder
# 3) MODEL_DIR
VIDEO_FILE = "0703.mp4"
VIDEO_PATH = None  # Set an absolute Path here only if automatic search fails.

# Full ensemble model selection.
# All listed models will be used for every selected video crop/frame.
ENSEMBLE_MODEL_FILES = [
    "yolov8s.pt",
    "yolo11s_norm750.pt",
    "yolo12s_norm750.pt",
    "yolov8s_norm1500.pt",
    "yolov10s_norm750.pt",
    "rtdetr-l.pt",
]

# Output folder and file names.
SCRIPT_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
OUT_ROOT = SCRIPT_DIR / "video_0703_zoom_ensemble_result"
OUTPUT_VIDEO_PATH = OUT_ROOT / "0703_zoom_layer_ensemble_detected.mp4"
OUTPUT_CSV_PATH = OUT_ROOT / "0703_zoom_layer_detections.csv"
OUTPUT_SUMMARY_PATH = OUT_ROOT / "0703_zoom_layer_video_summary.csv"
ASSIGNMENT_IMAGE_DIR = OUT_ROOT / "assignment_detection_images"


### Memory-safe 10% chunk saving settings


In [5]:
# ============================================================
# Memory-safe 10% chunk saving settings
# ============================================================
# When long 6-model ensemble video runs reach the late stage, keeping every
# detection row/summary row in Python lists can exhaust RAM.
# This mode saves CSV/video pieces every 10% of progress, then clears old
# in-memory rows and releases GPU cache.
ENABLE_10_PERCENT_CHUNK_SAVE = True
CHUNK_PERCENT_STEP = 10
CHUNK_ROOT = OUT_ROOT / "chunks_10_percent"
CHUNK_VIDEO_DIR = CHUNK_ROOT / "video_parts"
CHUNK_CSV_DIR = CHUNK_ROOT / "csv_parts"
CHUNK_MANIFEST_PATH = CHUNK_ROOT / "chunk_manifest.csv"

# Chunked video is safer than one huge VideoWriter session.
# Completed 10% video parts remain playable even if a later frame crashes.
SAVE_CHUNK_VIDEO_PARTS = True

# Keep this False for memory-safe assignment runs.
# The main result is saved as 10% video parts inside CHUNK_VIDEO_DIR.
SAVE_SINGLE_OUTPUT_VIDEO = False

# Extra cleanup cadence inside each chunk.
MEMORY_CLEANUP_EVERY_N_FRAMES = 5

# Model input image size.
# 1024 gives better detail, but is slower for 6-model ensemble video.
IMG_SIZE = 1024

# Prediction thresholds.
# RAW_MODEL_CONF_THRES is intentionally low so candidates are not removed too early.
RAW_MODEL_CONF_THRES = 0.03
PRED_CONF_THRES = 0.15
CONTAMINATION_CONF_THRES = 0.05
DAMAGE_CONF_THRES = 0.15

# Ensemble NMS removes duplicate boxes predicted by multiple models.
ENSEMBLE_NMS_IOU_THRES = 0.55

# Merge nearby damage/contamination boxes into one larger box for easier video review.
USE_MERGED_LARGE_BOXES_FOR_VIDEO = True
MERGE_GAP_RATIO = 0.08
MERGED_BOX_PADDING_RATIO = 0.02


### Zoom inspection layer settings


In [6]:
# ============================================================
# Zoom inspection layer settings
# ============================================================
# This patch does not use the old dataset/label verification flow.
# It inspects a zoomed turbine region from the drone video frame.
#
# ROI = region of interest.
# The script crops this turbine ROI, runs the full ensemble on the crop,
# then maps the detection boxes back onto the original video frame.

ENABLE_TURBINE_ZOOM_LAYER = True

# ROI mode:
# - "auto_follow": start from the center/manual ROI, then follow detected damage/contamination.
# - "manual": always use MANUAL_TURBINE_ROI_NORM.
# - "center": always use CENTER_TURBINE_ROI_NORM.
TURBINE_ROI_MODE = "auto_follow"

# Manual ROI format: (x1, y1, x2, y2), normalized 0.0 to 1.0.
# Example for a turbine in the middle-right area:
# MANUAL_TURBINE_ROI_NORM = (0.35, 0.10, 0.95, 0.85)
# Leave as None to use the center ROI first.
MANUAL_TURBINE_ROI_NORM = None

# Initial center crop used when manual ROI is not set.
# This large center area is usually reasonable for drone inspection footage.
CENTER_TURBINE_ROI_NORM = (0.12, 0.08, 0.88, 0.92)

# If no detection is found, periodically scan several zoom crops.
# This helps when the turbine/damage is small in the full drone image.
ENABLE_ZOOM_SEARCH_WHEN_NO_DETECTION = True
ZOOM_SEARCH_EVERY_N_FRAMES = 15
ZOOM_SEARCH_MAX_CANDIDATES = 5

# Candidate zoom crops used during search. Normalized xyxy.
# These cover center, left, right, upper, and lower areas.
SEARCH_ROIS_NORM = [
    (0.12, 0.08, 0.88, 0.92),  # center large
    (0.00, 0.05, 0.62, 0.95),  # left large
    (0.38, 0.05, 1.00, 0.95),  # right large
    (0.08, 0.00, 0.92, 0.62),  # upper large
    (0.08, 0.38, 0.92, 1.00),  # lower large
]

# When a defect/contamination is detected, expand the next ROI around it.
# Higher value means the zoom follows a wider turbine area, not only the tiny defect box.
TRACKING_ROI_EXPAND_RATIO = 1.80

# Minimum zoom crop size as a fraction of frame size.
# Prevents the ROI from becoming too tiny and losing turbine context.
MIN_ZOOM_ROI_WIDTH_RATIO = 0.28
MIN_ZOOM_ROI_HEIGHT_RATIO = 0.28

# ROI smoothing for video stability.
# 0.0 jumps immediately. 0.85 moves smoothly.
ROI_SMOOTHING = 0.82

# Optional full-frame detection.
# False is faster. The zoom-crop detections are still drawn back on the original frame.
# Set True if you also want the ensemble to inspect the whole frame every time.
RUN_FULL_FRAME_DETECTION_TOO = False

# Picture-in-picture zoom layer.
ZOOM_LAYER_WIDTH_RATIO = 0.38
ZOOM_LAYER_MARGIN = 18
ZOOM_LAYER_ALPHA_BACKGROUND = 0.82
DRAW_ROI_RECTANGLE_ON_ORIGINAL = True
DRAW_ZOOM_CONNECTOR_LINES = True

# Assignment still-image export.
# Saves detected overlay frames and zoom crops as separate JPG files.
SAVE_ASSIGNMENT_IMAGES = True
ASSIGNMENT_IMAGE_MIN_GAP_FRAMES = 30
ASSIGNMENT_IMAGE_MAX_COUNT = 40
SAVE_BEST_ASSIGNMENT_IMAGE = True

# Live preview.
# Many notebook/assignment/OpenCV-headless environments cannot use cv2.namedWindow()/imshow().
# Default is False so the script saves the result video/images without crashing.
# If your local OpenCV GUI works and you want a live window, set this to True.
SHOW_LIVE_PREVIEW = False
WINDOW_NAME = "0703.mp4 Zoom ROI Ensemble Detection"

# Save detection video and CSV.
SAVE_OUTPUT_VIDEO = True
SAVE_DETECTION_CSV = True

# Progress and debug options.
PRINT_EVERY_N_FRAMES = 10
MAX_FRAMES = None  # Set an integer for quick tests, e.g. 100. None means full video.

# Output codec.
# mp4v is widely available in OpenCV. If the output file is not playable, try "avc1".
OUTPUT_CODEC = "mp4v"
FORCE_OUTPUT_FPS = None  # None keeps source FPS. Set e.g. 30 if source FPS is missing.

# GPU options.
CUDA_AVAILABLE = torch is not None and getattr(torch, "cuda", None) is not None and torch.cuda.is_available()
DEVICE = 0 if CUDA_AVAILABLE else None
USE_HALF = CUDA_AVAILABLE


DAMAGE_KEYWORDS = [
    "damage", "defect", "crack", "erosion", "scratch", "dent", "burn", "broken", "fault"
]

CONTAMINATION_KEYWORDS = [
    "contamination", "contamin", "dirt", "dirty", "dust", "pollution", "pollut", "stain", "soiling"
]

MERGE_TARGET_KEYWORDS = DAMAGE_KEYWORDS + CONTAMINATION_KEYWORDS


### 1. BASIC UTILITIES


In [7]:
# ============================================================
# 1. BASIC UTILITIES
# ============================================================

def normalize_class_name(name: str) -> str:
    """Normalize class names for keyword matching."""
    return "".join(ch for ch in str(name).lower() if ch.isalnum())


def is_contamination_class(class_name: str) -> bool:
    """Return True for dirt/contamination-like model classes."""
    n = normalize_class_name(class_name)
    return any(k in n for k in CONTAMINATION_KEYWORDS)


def is_damage_class(class_name: str) -> bool:
    """Return True for damage/defect-like model classes."""
    n = normalize_class_name(class_name)
    return any(k in n for k in DAMAGE_KEYWORDS)


def is_merge_target_class(class_name: str) -> bool:
    """Return True if a class should use large-box merging."""
    n = normalize_class_name(class_name)
    return any(k in n for k in MERGE_TARGET_KEYWORDS)


def get_class_threshold(class_name: str) -> float:
    """Class-specific threshold after low-confidence raw model prediction."""
    if is_contamination_class(class_name):
        return CONTAMINATION_CONF_THRES
    if is_damage_class(class_name):
        return DAMAGE_CONF_THRES
    return PRED_CONF_THRES


def safe_imwrite(path: Path, image) -> None:
    """Save an image safely on Windows paths with Korean/non-ASCII characters."""
    path.parent.mkdir(parents=True, exist_ok=True)
    ext = path.suffix if path.suffix else ".jpg"
    ok, encoded = cv2.imencode(ext, image)
    if not ok:
        raise RuntimeError(f"Image encoding failed: {path}")
    encoded.tofile(str(path))


def box_iou_xyxy(a, b) -> float:
    """IoU for two boxes in xyxy format."""
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)
    inter = iw * ih

    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter

    return inter / union if union > 0 else 0.0


def clip_box_xyxy(box, img_w: int, img_h: int):
    """Clip a box to the frame boundary."""
    x1, y1, x2, y2 = box
    x1 = max(0.0, min(float(x1), float(img_w - 1)))
    x2 = max(0.0, min(float(x2), float(img_w - 1)))
    y1 = max(0.0, min(float(y1), float(img_h - 1)))
    y2 = max(0.0, min(float(y2), float(img_h - 1)))
    if x2 < x1:
        x1, x2 = x2, x1
    if y2 < y1:
        y1, y2 = y2, y1
    return x1, y1, x2, y2


def expand_box_xyxy(box, img_w: int, img_h: int, ratio: float):
    """Expand a box by a ratio of the frame's longer side."""
    x1, y1, x2, y2 = box
    pad = max(img_w, img_h) * ratio
    return clip_box_xyxy((x1 - pad, y1 - pad, x2 + pad, y2 + pad), img_w, img_h)


def boxes_intersect(a, b) -> bool:
    """Return True if two xyxy boxes intersect."""
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    return not (ax2 < bx1 or bx2 < ax1 or ay2 < by1 or by2 < ay1)


def union_box(boxes):
    """Return the smallest box containing all boxes."""
    x1 = min(b[0] for b in boxes)
    y1 = min(b[1] for b in boxes)
    x2 = max(b[2] for b in boxes)
    y2 = max(b[3] for b in boxes)
    return x1, y1, x2, y2


def seconds_to_hhmmss(seconds: float) -> str:
    """Format seconds for progress printing."""
    seconds = max(0.0, float(seconds))
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


### 2. MODEL LOADING


In [8]:
# ============================================================
# 2. MODEL LOADING
# ============================================================

def find_ensemble_model_paths():
    """Resolve all enabled ensemble model files into existing paths."""
    enabled_files = [str(name).strip() for name in ENSEMBLE_MODEL_FILES if str(name).strip()]
    if not enabled_files:
        raise RuntimeError("No ensemble model files are enabled.")

    resolved_paths = []
    for name in enabled_files:
        raw_path = Path(name).expanduser()
        candidates = []

        if raw_path.is_absolute():
            candidates.append(raw_path)
        else:
            candidates.extend([
                MODEL_DIR / name,
                SCRIPT_DIR / name,
                Path.cwd() / name,
            ])

        found = None
        for candidate in candidates:
            if candidate.exists() and candidate.is_file():
                found = candidate.resolve()
                break

        if found is None:
            checked = "\n".join(str(p) for p in candidates)
            raise FileNotFoundError(
                f"Could not find ensemble model file: {name}\n"
                f"Checked these paths:\n{checked}"
            )

        resolved_paths.append(found)

    unique_paths = []
    seen = set()
    for path in resolved_paths:
        key = str(path)
        if key not in seen:
            unique_paths.append(path)
            seen.add(key)

    return unique_paths


def load_ensemble_models():
    """Load all ensemble models and verify matching class order."""
    model_paths = find_ensemble_model_paths()
    models = []

    print("\n" + "=" * 70)
    print("LOAD ENSEMBLE MODEL(S)")
    print("=" * 70)
    print("CUDA available:", CUDA_AVAILABLE)
    print("Device        :", DEVICE if DEVICE is not None else "CPU")
    print("Half precision:", USE_HALF)

    for path in model_paths:
        print("Loading:", path)
        model = YOLO(str(path))
        try:
            model.fuse()
        except Exception:
            pass
        models.append(model)

    model_labels = [path.name for path in model_paths]

    reference_names = [models[0].names[i] for i in sorted(models[0].names.keys())]
    for model_path, model in zip(model_paths[1:], models[1:]):
        names = [model.names[i] for i in sorted(model.names.keys())]
        if names != reference_names:
            raise RuntimeError(
                "Ensemble model class order mismatch. All models must have identical class IDs/names.\n"
                f"Reference ({model_paths[0].name}): {reference_names}\n"
                f"Mismatch  ({model_path.name}): {names}"
            )

    print("\nLoaded ensemble models:")
    for i, path in enumerate(model_paths, start=1):
        print(f"- [{i}/{len(model_paths)}] {path.name}")

    print("\nModel classes:")
    for idx, name in enumerate(reference_names):
        print(f"- {idx}: {name}")

    return models, model_paths, model_labels, reference_names


### 3. ENSEMBLE PREDICTION AND POST-PROCESSING


In [9]:
# ============================================================
# 3. ENSEMBLE PREDICTION AND POST-PROCESSING
# ============================================================

def get_prediction_boxes_from_result(result, class_names, source_model="model"):
    """Convert one Ultralytics result.boxes object into plain dictionaries."""
    pred_boxes = []
    if result.boxes is None or len(result.boxes) == 0:
        return pred_boxes

    for i in range(len(result.boxes)):
        cls_id = int(result.boxes.cls[i].item())
        conf = float(result.boxes.conf[i].item())
        x1, y1, x2, y2 = result.boxes.xyxy[i].tolist()
        class_name = class_names[cls_id] if 0 <= cls_id < len(class_names) else str(cls_id)

        pred_boxes.append({
            "class_id": cls_id,
            "class_name": class_name,
            "confidence": conf,
            "xmin": float(x1),
            "ymin": float(y1),
            "xmax": float(x2),
            "ymax": float(y2),
            "source": "raw_ensemble",
            "source_model": source_model,
            "source_models": source_model,
            "ensemble_support_count": 1,
            "merged_from_count": 1,
        })

    return pred_boxes


def nms_ensemble_boxes(pred_boxes, iou_thres=0.55):
    """Class-wise NMS for pooled boxes from all ensemble models."""
    if not pred_boxes:
        return []

    kept = []
    boxes_by_class = defaultdict(list)
    for box in pred_boxes:
        boxes_by_class[int(box["class_id"])].append(dict(box))

    for _, class_boxes in boxes_by_class.items():
        remaining = sorted(class_boxes, key=lambda x: float(x["confidence"]), reverse=True)

        while remaining:
            best = remaining.pop(0)
            best_box = (best["xmin"], best["ymin"], best["xmax"], best["ymax"])
            still_remaining = []

            supporting_models = set()
            for field in ("source_model", "source_models"):
                for model_name in str(best.get(field, "")).split(","):
                    model_name = model_name.strip()
                    if model_name:
                        supporting_models.add(model_name)

            for other in remaining:
                other_box = (other["xmin"], other["ymin"], other["xmax"], other["ymax"])
                if box_iou_xyxy(best_box, other_box) >= iou_thres:
                    for field in ("source_model", "source_models"):
                        for model_name in str(other.get(field, "")).split(","):
                            model_name = model_name.strip()
                            if model_name:
                                supporting_models.add(model_name)
                else:
                    still_remaining.append(other)

            best["source"] = "ensemble_nms"
            best["source_models"] = ",".join(sorted(supporting_models))
            best["ensemble_support_count"] = len(supporting_models)
            kept.append(best)
            remaining = still_remaining

    return sorted(kept, key=lambda x: float(x["confidence"]), reverse=True)


def filter_pred_boxes_by_class_threshold(pred_boxes):
    """Apply class-specific confidence thresholds after low-threshold prediction."""
    kept = []
    dropped_counter = Counter()

    for box in pred_boxes:
        threshold = get_class_threshold(box["class_name"])
        if float(box["confidence"]) >= threshold:
            box["class_threshold_used"] = threshold
            kept.append(box)
        else:
            dropped_counter[box["class_name"]] += 1

    return kept, dropped_counter


def merge_large_damage_boxes(pred_boxes, image_shape):
    """Merge nearby damage/contamination boxes into larger review boxes."""
    if not USE_MERGED_LARGE_BOXES_FOR_VIDEO:
        return pred_boxes

    img_h, img_w = image_shape[:2]
    keep_boxes = []
    merge_candidates_by_class = defaultdict(list)

    for box in pred_boxes:
        if is_merge_target_class(box["class_name"]):
            merge_candidates_by_class[box["class_id"]].append(box)
        else:
            keep_boxes.append(box)

    merged_boxes = []

    for cls_id, boxes in merge_candidates_by_class.items():
        clusters = []

        for box in sorted(boxes, key=lambda x: x["confidence"], reverse=True):
            b_box = (box["xmin"], box["ymin"], box["xmax"], box["ymax"])
            b_expanded = expand_box_xyxy(b_box, img_w, img_h, MERGE_GAP_RATIO)

            matched_cluster_idx = None
            for ci, cluster in enumerate(clusters):
                cluster_box = union_box([(x["xmin"], x["ymin"], x["xmax"], x["ymax"]) for x in cluster])
                cluster_expanded = expand_box_xyxy(cluster_box, img_w, img_h, MERGE_GAP_RATIO)
                if boxes_intersect(b_expanded, cluster_expanded):
                    matched_cluster_idx = ci
                    break

            if matched_cluster_idx is None:
                clusters.append([box])
            else:
                clusters[matched_cluster_idx].append(box)

        changed = True
        while changed:
            changed = False
            new_clusters = []

            for cluster in clusters:
                c_box = union_box([(x["xmin"], x["ymin"], x["xmax"], x["ymax"]) for x in cluster])
                c_expanded = expand_box_xyxy(c_box, img_w, img_h, MERGE_GAP_RATIO)

                merged_into_existing = False
                for existing in new_clusters:
                    e_box = union_box([(x["xmin"], x["ymin"], x["xmax"], x["ymax"]) for x in existing])
                    e_expanded = expand_box_xyxy(e_box, img_w, img_h, MERGE_GAP_RATIO)
                    if boxes_intersect(c_expanded, e_expanded):
                        existing.extend(cluster)
                        changed = True
                        merged_into_existing = True
                        break

                if not merged_into_existing:
                    new_clusters.append(cluster)

            clusters = new_clusters

        for cluster in clusters:
            raw_union = union_box([(x["xmin"], x["ymin"], x["xmax"], x["ymax"]) for x in cluster])
            x1, y1, x2, y2 = expand_box_xyxy(raw_union, img_w, img_h, MERGED_BOX_PADDING_RATIO)
            best = max(cluster, key=lambda x: x["confidence"])
            mean_conf = sum(x["confidence"] for x in cluster) / len(cluster)

            cluster_source_models = set()
            for item in cluster:
                for model_name in str(item.get("source_models", item.get("source_model", ""))).split(","):
                    model_name = model_name.strip()
                    if model_name:
                        cluster_source_models.add(model_name)

            merged_boxes.append({
                "class_id": cls_id,
                "class_name": best["class_name"],
                "confidence": float(best["confidence"]),
                "mean_confidence": float(mean_conf),
                "xmin": x1,
                "ymin": y1,
                "xmax": x2,
                "ymax": y2,
                "source": "merged_largebox" if len(cluster) > 1 else "single_largebox",
                "source_models": ",".join(sorted(cluster_source_models)),
                "ensemble_support_count": len(cluster_source_models),
                "merged_from_count": len(cluster),
                "class_threshold_used": best.get("class_threshold_used", get_class_threshold(best["class_name"])),
            })

    return keep_boxes + merged_boxes


def predict_with_ensemble_on_frame(models, model_labels, frame_bgr, class_names, imgsz: int, conf: float):
    """Run all ensemble models on one BGR image/crop, then apply ensemble NMS."""
    pooled_boxes = []

    for model, model_label in zip(models, model_labels):
        predict_kwargs = {
            "source": frame_bgr,
            "imgsz": imgsz,
            "conf": conf,
            "verbose": False,
        }
        if DEVICE is not None:
            predict_kwargs["device"] = DEVICE
        if USE_HALF:
            predict_kwargs["half"] = True

        result = model.predict(**predict_kwargs)[0]
        pooled_boxes.extend(get_prediction_boxes_from_result(result, class_names, source_model=model_label))

    return nms_ensemble_boxes(pooled_boxes, iou_thres=ENSEMBLE_NMS_IOU_THRES)


def run_postprocess_pipeline(raw_pred_boxes, image_shape):
    """Apply class thresholds and optional large-box merge."""
    filtered_boxes, dropped_counter = filter_pred_boxes_by_class_threshold(raw_pred_boxes)
    final_boxes = merge_large_damage_boxes(filtered_boxes, image_shape)
    final_boxes = sorted(final_boxes, key=lambda x: float(x["confidence"]), reverse=True)
    return final_boxes, filtered_boxes, dropped_counter


### 4. ZOOM ROI UTILITIES


In [10]:
# ============================================================
# 4. ZOOM ROI UTILITIES
# ============================================================

def norm_roi_to_xyxy(norm_roi, width: int, height: int):
    """Convert normalized xyxy ROI to absolute xyxy coordinates."""
    x1, y1, x2, y2 = norm_roi
    return sanitize_roi((x1 * width, y1 * height, x2 * width, y2 * height), width, height)


def sanitize_roi(roi, width: int, height: int):
    """Clip and validate an ROI."""
    x1, y1, x2, y2 = clip_box_xyxy(roi, width, height)
    if x2 - x1 < 4 or y2 - y1 < 4:
        return 0.0, 0.0, float(width - 1), float(height - 1)
    return x1, y1, x2, y2


def ensure_min_roi_size(roi, width: int, height: int):
    """Expand ROI to a minimum width/height while preserving center."""
    x1, y1, x2, y2 = sanitize_roi(roi, width, height)
    cx = (x1 + x2) / 2.0
    cy = (y1 + y2) / 2.0

    min_w = width * MIN_ZOOM_ROI_WIDTH_RATIO
    min_h = height * MIN_ZOOM_ROI_HEIGHT_RATIO
    roi_w = max(x2 - x1, min_w)
    roi_h = max(y2 - y1, min_h)

    new_roi = (cx - roi_w / 2.0, cy - roi_h / 2.0, cx + roi_w / 2.0, cy + roi_h / 2.0)
    return sanitize_roi(new_roi, width, height)


def smooth_roi(prev_roi, new_roi, width: int, height: int):
    """Smooth ROI movement for a steadier video layer."""
    if prev_roi is None or ROI_SMOOTHING <= 0:
        return sanitize_roi(new_roi, width, height)

    a = float(ROI_SMOOTHING)
    smoothed = tuple(a * float(p) + (1.0 - a) * float(n) for p, n in zip(prev_roi, new_roi))
    return sanitize_roi(smoothed, width, height)


def get_initial_roi(width: int, height: int):
    """Get initial turbine ROI from manual setting or center default."""
    if MANUAL_TURBINE_ROI_NORM is not None:
        return norm_roi_to_xyxy(MANUAL_TURBINE_ROI_NORM, width, height)
    return norm_roi_to_xyxy(CENTER_TURBINE_ROI_NORM, width, height)


def get_search_rois(width: int, height: int):
    """Build candidate ROIs for periodic zoom search."""
    rois = []
    for norm_roi in SEARCH_ROIS_NORM[: int(ZOOM_SEARCH_MAX_CANDIDATES)]:
        rois.append(norm_roi_to_xyxy(norm_roi, width, height))
    return rois


def crop_frame_by_roi(frame, roi):
    """Crop a frame by absolute xyxy ROI."""
    h, w = frame.shape[:2]
    x1, y1, x2, y2 = sanitize_roi(roi, w, h)
    ix1, iy1, ix2, iy2 = map(int, [x1, y1, x2, y2])
    crop = frame[iy1:iy2 + 1, ix1:ix2 + 1].copy()
    return crop, (float(ix1), float(iy1), float(ix2), float(iy2))


def local_boxes_to_global(local_boxes, roi):
    """Map crop-local boxes back to original frame coordinates."""
    rx1, ry1, _, _ = roi
    global_boxes = []
    for box in local_boxes:
        b = dict(box)
        b["xmin"] = float(box["xmin"]) + rx1
        b["xmax"] = float(box["xmax"]) + rx1
        b["ymin"] = float(box["ymin"]) + ry1
        b["ymax"] = float(box["ymax"]) + ry1
        b["source"] = "zoom_roi_" + str(b.get("source", "ensemble"))
        b["roi_xmin"] = rx1
        b["roi_ymin"] = ry1
        b["roi_xmax"] = roi[2]
        b["roi_ymax"] = roi[3]
        global_boxes.append(b)
    return global_boxes


def make_tracking_roi_from_boxes(global_boxes, width: int, height: int):
    """Make next ROI around detected boxes with generous turbine context."""
    if not global_boxes:
        return None
    boxes = [(b["xmin"], b["ymin"], b["xmax"], b["ymax"]) for b in global_boxes]
    x1, y1, x2, y2 = union_box(boxes)

    box_w = max(4.0, x2 - x1)
    box_h = max(4.0, y2 - y1)
    pad_x = box_w * TRACKING_ROI_EXPAND_RATIO
    pad_y = box_h * TRACKING_ROI_EXPAND_RATIO

    roi = (x1 - pad_x, y1 - pad_y, x2 + pad_x, y2 + pad_y)
    roi = ensure_min_roi_size(roi, width, height)
    return sanitize_roi(roi, width, height)


def should_run_search(frame_index: int, last_detection_frame: int) -> bool:
    """Decide whether to scan multiple zoom crops."""
    if not ENABLE_ZOOM_SEARCH_WHEN_NO_DETECTION:
        return False
    if TURBINE_ROI_MODE == "manual":
        return False
    if frame_index == 1:
        return True
    if last_detection_frame <= 0:
        return frame_index % int(ZOOM_SEARCH_EVERY_N_FRAMES) == 0
    frames_since_detection = frame_index - last_detection_frame
    return frames_since_detection >= int(ZOOM_SEARCH_EVERY_N_FRAMES) and frame_index % int(ZOOM_SEARCH_EVERY_N_FRAMES) == 0


def score_roi_detection(final_boxes) -> float:
    """Score ROI candidates by confidence and number of detections."""
    if not final_boxes:
        return 0.0
    max_conf = max(float(b["confidence"]) for b in final_boxes)
    support_sum = sum(int(b.get("ensemble_support_count", 1)) for b in final_boxes)
    return max_conf * 10.0 + len(final_boxes) + support_sum * 0.1


def inspect_roi_with_ensemble(models, model_labels, class_names, frame, roi, roi_name):
    """Crop one ROI, run ensemble on it, and return local/global detections."""
    crop, abs_roi = crop_frame_by_roi(frame, roi)
    raw_local_boxes = predict_with_ensemble_on_frame(
        models=models,
        model_labels=model_labels,
        frame_bgr=crop,
        class_names=class_names,
        imgsz=IMG_SIZE,
        conf=RAW_MODEL_CONF_THRES,
    )
    local_boxes, filtered_boxes, dropped_counter = run_postprocess_pipeline(raw_local_boxes, crop.shape)
    global_boxes = local_boxes_to_global(local_boxes, abs_roi)

    for box in local_boxes:
        box["roi_name"] = roi_name
    for box in global_boxes:
        box["roi_name"] = roi_name

    return {
        "roi_name": roi_name,
        "roi": abs_roi,
        "crop": crop,
        "raw_local_boxes": raw_local_boxes,
        "filtered_local_boxes": filtered_boxes,
        "local_boxes": local_boxes,
        "global_boxes": global_boxes,
        "dropped_counter": dropped_counter,
        "score": score_roi_detection(local_boxes),
    }


def choose_zoom_inspection(models, model_labels, class_names, frame, frame_index, current_roi, last_detection_frame):
    """Choose and inspect the best zoom ROI for this frame."""
    h, w = frame.shape[:2]

    if TURBINE_ROI_MODE == "manual":
        base_roi = norm_roi_to_xyxy(MANUAL_TURBINE_ROI_NORM or CENTER_TURBINE_ROI_NORM, w, h)
    elif TURBINE_ROI_MODE == "center":
        base_roi = norm_roi_to_xyxy(CENTER_TURBINE_ROI_NORM, w, h)
    else:
        base_roi = current_roi if current_roi is not None else get_initial_roi(w, h)

    candidate_rois = [("tracked_zoom", base_roi)]

    if should_run_search(frame_index, last_detection_frame):
        candidate_rois = [("search_center", get_initial_roi(w, h))]
        for idx, roi in enumerate(get_search_rois(w, h), start=1):
            candidate_rois.append((f"search_{idx}", roi))

    # Remove duplicate ROIs after rounding so the same crop is not inspected twice.
    deduped = []
    seen = set()
    for name, roi in candidate_rois:
        key = tuple(int(round(v)) for v in roi)
        if key not in seen:
            seen.add(key)
            deduped.append((name, roi))

    results = []
    for name, roi in deduped:
        results.append(inspect_roi_with_ensemble(models, model_labels, class_names, frame, roi, name))

    best = max(results, key=lambda r: r["score"]) if results else None
    if best is None:
        return None

    if best["global_boxes"] and TURBINE_ROI_MODE == "auto_follow":
        next_roi = make_tracking_roi_from_boxes(best["global_boxes"], w, h)
        next_roi = smooth_roi(current_roi, next_roi, w, h)
    else:
        next_roi = smooth_roi(current_roi, best["roi"], w, h) if TURBINE_ROI_MODE == "auto_follow" else best["roi"]

    best["next_roi"] = next_roi
    best["candidate_count"] = len(deduped)
    return best


### 5. DRAWING AND VIDEO IO


In [11]:
# ============================================================
# 5. DRAWING AND VIDEO IO
# ============================================================

def draw_prediction_boxes_from_list(image_bgr, pred_boxes, title_prefix="PRED", box_color=(255, 255, 0)):
    """Draw post-processed prediction boxes on a frame."""
    img = image_bgr.copy()
    img_h, img_w = img.shape[:2]

    for box in pred_boxes:
        x1, y1, x2, y2 = clip_box_xyxy(
            (box["xmin"], box["ymin"], box["xmax"], box["ymax"]),
            img_w,
            img_h,
        )
        x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])

        cls_name = box["class_name"]
        conf = float(box["confidence"])
        support = int(box.get("ensemble_support_count", 1))
        merged_n = int(box.get("merged_from_count", 1))

        label = f"{title_prefix} {cls_name} {conf:.2f} ens{support}"
        if merged_n > 1:
            label += f" x{merged_n}"

        cv2.rectangle(img, (x1, y1), (x2, y2), box_color, 3)

        text_y = max(24, y1 - 8)
        cv2.putText(
            img,
            label,
            (x1, text_y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            box_color,
            2,
            lineType=cv2.LINE_AA,
        )

    return img


def draw_frame_info(frame, frame_index, total_frames, detection_count, zoom_detection_count, fps_now, elapsed_sec, roi_name):
    """Draw frame progress information in the corner."""
    img = frame.copy()
    total_text = str(total_frames) if total_frames and total_frames > 0 else "?"
    text = (
        f"frame {frame_index}/{total_text} | detections {detection_count} | "
        f"zoom detections {zoom_detection_count} | {roi_name} | "
        f"speed {fps_now:.2f} FPS | elapsed {seconds_to_hhmmss(elapsed_sec)}"
    )

    cv2.rectangle(img, (8, 8), (min(img.shape[1] - 8, 1180), 46), (0, 0, 0), -1)
    cv2.putText(
        img,
        text,
        (18, 34),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.62,
        (255, 255, 255),
        2,
        lineType=cv2.LINE_AA,
    )
    return img


def draw_roi_rectangle(frame, roi, label="TURBINE ZOOM ROI"):
    """Draw the turbine zoom ROI rectangle on the original frame."""
    img = frame.copy()
    h, w = img.shape[:2]
    x1, y1, x2, y2 = sanitize_roi(roi, w, h)
    x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])

    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 210, 255), 3)
    cv2.putText(
        img,
        label,
        (x1, max(28, y1 - 10)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.72,
        (0, 210, 255),
        2,
        lineType=cv2.LINE_AA,
    )
    return img


def add_text_panel(image, title, subtitle=None):
    """Add a readable title strip to a crop/panel."""
    img = image.copy()
    h, w = img.shape[:2]
    panel_h = 46 if subtitle is None else 72
    overlay = img.copy()
    cv2.rectangle(overlay, (0, 0), (w, panel_h), (0, 0, 0), -1)
    img = cv2.addWeighted(overlay, 0.70, img, 0.30, 0)

    cv2.putText(img, title, (12, 29), cv2.FONT_HERSHEY_SIMPLEX, 0.78, (255, 255, 255), 2, lineType=cv2.LINE_AA)
    if subtitle:
        cv2.putText(img, subtitle, (12, 58), cv2.FONT_HERSHEY_SIMPLEX, 0.58, (255, 255, 255), 2, lineType=cv2.LINE_AA)
    return img


def overlay_zoom_layer(frame, zoom_crop_drawn, roi, detection_count, roi_name):
    """Overlay the zoomed turbine crop as a picture-in-picture layer."""
    if zoom_crop_drawn is None or zoom_crop_drawn.size == 0:
        return frame

    img = frame.copy()
    h, w = img.shape[:2]

    layer_w = int(w * float(ZOOM_LAYER_WIDTH_RATIO))
    layer_w = max(260, min(layer_w, w - 2 * ZOOM_LAYER_MARGIN))
    crop_h, crop_w = zoom_crop_drawn.shape[:2]
    layer_h = int(layer_w * crop_h / max(1, crop_w))
    max_layer_h = h - 2 * ZOOM_LAYER_MARGIN
    if layer_h > max_layer_h:
        layer_h = max_layer_h
        layer_w = int(layer_h * crop_w / max(1, crop_h))

    resized = cv2.resize(zoom_crop_drawn, (layer_w, layer_h))
    title = "ZOOM-IN TURBINE INSPECTION"
    subtitle = f"ROI: {roi_name} | detected: {detection_count}"
    resized = add_text_panel(resized, title, subtitle)

    x2 = w - ZOOM_LAYER_MARGIN
    y1 = ZOOM_LAYER_MARGIN
    x1 = x2 - layer_w
    y2 = y1 + layer_h

    # Background shadow panel.
    overlay = img.copy()
    cv2.rectangle(overlay, (x1 - 8, y1 - 8), (x2 + 8, y2 + 8), (0, 0, 0), -1)
    img = cv2.addWeighted(overlay, ZOOM_LAYER_ALPHA_BACKGROUND, img, 1.0 - ZOOM_LAYER_ALPHA_BACKGROUND, 0)

    img[y1:y2, x1:x2] = resized
    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 210, 255), 3)

    if DRAW_ZOOM_CONNECTOR_LINES:
        rx1, ry1, rx2, ry2 = map(int, roi)
        cv2.line(img, (rx2, ry1), (x1, y1), (0, 210, 255), 2, lineType=cv2.LINE_AA)
        cv2.line(img, (rx2, ry2), (x1, y2), (0, 210, 255), 2, lineType=cv2.LINE_AA)

    return img


def resolve_video_path() -> Path:
    """Find 0702.mp4 in the same folder as script, cwd, or MODEL_DIR."""
    if VIDEO_PATH is not None:
        p = Path(VIDEO_PATH).expanduser()
        if p.exists() and p.is_file():
            return p.resolve()
        raise FileNotFoundError(f"VIDEO_PATH was set but the file was not found: {p}")

    candidates = [
        SCRIPT_DIR / VIDEO_FILE,
        Path.cwd() / VIDEO_FILE,
        MODEL_DIR / VIDEO_FILE,
    ]

    for candidate in candidates:
        if candidate.exists() and candidate.is_file():
            return candidate.resolve()

    checked = "\n".join(str(p) for p in candidates)
    raise FileNotFoundError(
        f"Could not find {VIDEO_FILE}.\n"
        f"Put it in the same folder as this script or in MODEL_DIR.\n"
        f"Checked these paths:\n{checked}"
    )


def open_video_capture(video_path: Path):
    """Open video with OpenCV."""
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video file: {video_path}")
    return cap


def make_video_writer(output_path: Path, fps: float, width: int, height: int):
    """Create an OpenCV VideoWriter for the result video."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fourcc = cv2.VideoWriter_fourcc(*OUTPUT_CODEC)
    writer = cv2.VideoWriter(str(output_path), fourcc, fps, (width, height))

    if not writer.isOpened():
        raise RuntimeError(
            f"Could not open VideoWriter: {output_path}\n"
            f"Try changing OUTPUT_CODEC from {OUTPUT_CODEC!r} to 'avc1' or 'XVID'."
        )

    return writer


def try_start_live_preview() -> bool:
    """Start OpenCV live preview only when the installed OpenCV supports GUI windows."""
    if not SHOW_LIVE_PREVIEW:
        return False

    try:
        cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)
        return True
    except cv2.error as e:
        print("\nWARNING: OpenCV GUI preview is not available in this environment.")
        print("The script will continue without cv2.imshow().")
        print("The output video and assignment images will still be saved normally.")
        print("OpenCV error:", e)
        return False


def show_live_preview_frame(frame) -> bool:
    """Show one preview frame. Return False when preview should be disabled/stopped."""
    try:
        cv2.imshow(WINDOW_NAME, frame)
        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            print("\nStopped early by user pressing q.")
            return False
        return True
    except cv2.error as e:
        print("\nWARNING: cv2.imshow() failed. Disabling live preview and continuing save-only mode.")
        print("OpenCV error:", e)
        return False


def safely_close_live_preview(preview_enabled: bool):
    """Close OpenCV preview windows only if preview was actually enabled."""
    if not preview_enabled:
        return
    try:
        cv2.destroyAllWindows()
    except cv2.error:
        pass


def maybe_save_assignment_images(frame_index, plotted_frame, zoom_crop_drawn, zoom_boxes, assignment_state):
    """Save selected still images for assignment submission."""
    if not SAVE_ASSIGNMENT_IMAGES:
        return
    if not zoom_boxes:
        return

    max_conf = max(float(b["confidence"]) for b in zoom_boxes)

    if SAVE_BEST_ASSIGNMENT_IMAGE and max_conf > assignment_state.get("best_conf", -1.0):
        assignment_state["best_conf"] = max_conf
        safe_imwrite(ASSIGNMENT_IMAGE_DIR / "best_overlay_frame.jpg", plotted_frame)
        if zoom_crop_drawn is not None and zoom_crop_drawn.size > 0:
            safe_imwrite(ASSIGNMENT_IMAGE_DIR / "best_zoom_crop.jpg", zoom_crop_drawn)

    if assignment_state.get("saved_count", 0) >= ASSIGNMENT_IMAGE_MAX_COUNT:
        return

    last_frame = assignment_state.get("last_saved_frame", -10**9)
    if frame_index - last_frame < ASSIGNMENT_IMAGE_MIN_GAP_FRAMES:
        return

    ASSIGNMENT_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
    overlay_path = ASSIGNMENT_IMAGE_DIR / f"frame_{frame_index:06d}_overlay.jpg"
    zoom_path = ASSIGNMENT_IMAGE_DIR / f"frame_{frame_index:06d}_zoom_crop.jpg"
    safe_imwrite(overlay_path, plotted_frame)
    if zoom_crop_drawn is not None and zoom_crop_drawn.size > 0:
        safe_imwrite(zoom_path, zoom_crop_drawn)

    assignment_state["last_saved_frame"] = frame_index
    assignment_state["saved_count"] = assignment_state.get("saved_count", 0) + 1


### 6. MEMORY-SAFE CHUNK UTILITIES


In [12]:
# ============================================================
# 6. MEMORY-SAFE CHUNK UTILITIES
# ============================================================

def cleanup_runtime_memory(reason=""):
    """Release Python and CUDA temporary memory after chunk/frame processing."""
    gc.collect()
    if torch is not None and CUDA_AVAILABLE:
        try:
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        except Exception:
            pass
    if reason:
        print(f"[MEMORY CLEANUP] {reason}")


def get_progress_chunk_info(frame_index: int, total_frames: int):
    """Return chunk index and percentage range for the current frame."""
    step = max(1, int(CHUNK_PERCENT_STEP))

    if total_frames and total_frames > 0:
        # frame_index is 1-based. Use frame_index - 1 so frame 1 starts in 0-10%.
        progress_before = ((frame_index - 1) / float(total_frames)) * 100.0
        chunk_index = int(progress_before // step)
        chunk_index = max(0, min(chunk_index, int(np.ceil(100 / step)) - 1))
        start_pct = chunk_index * step
        end_pct = min(100, start_pct + step)
    else:
        # Fallback when frame count is unknown.
        fallback_frames = 100
        chunk_index = int((frame_index - 1) // fallback_frames)
        start_pct = chunk_index * step
        end_pct = start_pct + step

    return {
        "chunk_index": chunk_index,
        "start_pct": int(start_pct),
        "end_pct": int(end_pct),
        "chunk_name": f"{int(start_pct):03d}_{int(end_pct):03d}",
    }


def make_chunk_video_path(chunk_info):
    """Build a 10% chunk video output path."""
    stem = Path(VIDEO_FILE).stem
    return CHUNK_VIDEO_DIR / f"{stem}_zoom_layer_part_{chunk_info['chunk_name']}.mp4"


def open_chunk_video_writer(chunk_info, fps: float, width: int, height: int):
    """Open a video writer for one 10% chunk."""
    if not SAVE_OUTPUT_VIDEO or not SAVE_CHUNK_VIDEO_PARTS:
        return None, ""

    CHUNK_VIDEO_DIR.mkdir(parents=True, exist_ok=True)
    chunk_path = make_chunk_video_path(chunk_info)
    writer = make_video_writer(chunk_path, fps, width, height)
    print(f"[CHUNK VIDEO OPEN] {chunk_info['chunk_name']} -> {chunk_path}")
    return writer, str(chunk_path)


def flush_chunk_rows(detection_rows, summary_rows, chunk_info, chunk_manifest_rows, reason="chunk_end"):
    """Save accumulated detection/summary rows for one chunk, then clear RAM lists."""
    if not SAVE_DETECTION_CSV:
        detection_rows.clear()
        summary_rows.clear()
        cleanup_runtime_memory(f"{reason}: cleared rows without CSV")
        return

    CHUNK_CSV_DIR.mkdir(parents=True, exist_ok=True)
    chunk_name = chunk_info.get("chunk_name", "unknown")
    det_path = CHUNK_CSV_DIR / f"detections_part_{chunk_name}.csv"
    sum_path = CHUNK_CSV_DIR / f"summary_part_{chunk_name}.csv"

    det_count = len(detection_rows)
    sum_count = len(summary_rows)

    if det_count > 0:
        pd.DataFrame(detection_rows).to_csv(det_path, index=False, encoding="utf-8-sig")
    else:
        # Keep an empty marker file so every chunk has a matching artifact.
        pd.DataFrame().to_csv(det_path, index=False, encoding="utf-8-sig")

    if sum_count > 0:
        pd.DataFrame(summary_rows).to_csv(sum_path, index=False, encoding="utf-8-sig")
    else:
        pd.DataFrame().to_csv(sum_path, index=False, encoding="utf-8-sig")

    chunk_manifest_rows.append({
        "chunk_index": int(chunk_info.get("chunk_index", -1)),
        "chunk_name": chunk_name,
        "start_pct": int(chunk_info.get("start_pct", -1)),
        "end_pct": int(chunk_info.get("end_pct", -1)),
        "reason": reason,
        "detection_rows_saved": det_count,
        "summary_rows_saved": sum_count,
        "detection_csv": str(det_path),
        "summary_csv": str(sum_path),
        "video_part": chunk_info.get("video_part", ""),
    })

    detection_rows.clear()
    summary_rows.clear()
    cleanup_runtime_memory(f"{reason}: saved {chunk_name}, det_rows={det_count}, summary_rows={sum_count}")


def save_chunk_manifest(chunk_manifest_rows):
    """Save the list of all produced 10% chunks plus an ffmpeg concat list."""
    CHUNK_ROOT.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(chunk_manifest_rows).to_csv(CHUNK_MANIFEST_PATH, index=False, encoding="utf-8-sig")

    concat_list_path = CHUNK_ROOT / "video_concat_list.txt"
    video_parts = []
    for row in chunk_manifest_rows:
        video_part = str(row.get("video_part", "")).strip()
        if video_part:
            video_parts.append(video_part)

    # Remove duplicates while preserving order.
    deduped_parts = []
    seen = set()
    for path in video_parts:
        if path not in seen:
            seen.add(path)
            deduped_parts.append(path)

    with open(concat_list_path, "w", encoding="utf-8") as f:
        for path in deduped_parts:
            # ffmpeg concat demuxer accepts forward slashes more reliably on Windows.
            safe_path = str(Path(path)).replace("\\", "/")
            f.write(f"file '{safe_path}'\n")

    print("Saved chunk manifest:", CHUNK_MANIFEST_PATH)
    print("Saved ffmpeg concat list:", concat_list_path)


### 6. MAIN VIDEO DETECTION PIPELINE


In [13]:
# ============================================================
# 6. MAIN VIDEO DETECTION PIPELINE
# ============================================================

def run_video_detection():
    """Run full ensemble zoom inspection on 0702.mp4 and save the boxed video."""
    print("=" * 70)
    print("0702.MP4 ZOOM ROI ENSEMBLE VIDEO DETECTION")
    print("=" * 70)
    print("This script does NOT build or validate any dataset.")
    print("It reads 0702.mp4, crops/zooms the turbine region, runs the full ensemble, draws boxes, and saves the result video.")

    OUT_ROOT.mkdir(parents=True, exist_ok=True)
    ASSIGNMENT_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

    video_path = resolve_video_path()
    print("\nVideo source:", video_path)
    print("Output video:", OUTPUT_VIDEO_PATH if SAVE_SINGLE_OUTPUT_VIDEO else "chunked video parts")
    print("Chunk videos:", CHUNK_VIDEO_DIR)
    print("Chunk CSVs  :", CHUNK_CSV_DIR)
    print("Assignment images:", ASSIGNMENT_IMAGE_DIR)

    models, model_paths, model_labels, class_names = load_ensemble_models()

    cap = open_video_capture(video_path)

    src_fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)

    if width <= 0 or height <= 0:
        cap.release()
        raise RuntimeError("Could not read video width/height from the source video.")

    out_fps = float(FORCE_OUTPUT_FPS) if FORCE_OUTPUT_FPS is not None else src_fps
    if out_fps <= 0:
        out_fps = 30.0

    print("\n" + "=" * 70)
    print("VIDEO INFO")
    print("=" * 70)
    print("Width x Height:", width, "x", height)
    print("Source FPS    :", src_fps)
    print("Output FPS    :", out_fps)
    print("Total frames  :", total_frames if total_frames > 0 else "unknown")
    print("IMG_SIZE      :", IMG_SIZE)
    print("Raw conf      :", RAW_MODEL_CONF_THRES)
    print("Default conf  :", PRED_CONF_THRES)
    print("Damage conf   :", DAMAGE_CONF_THRES)
    print("Contam. conf  :", CONTAMINATION_CONF_THRES)
    print("Ensemble NMS  :", ENSEMBLE_NMS_IOU_THRES)
    print("Models        :", len(models))
    print("ROI mode      :", TURBINE_ROI_MODE)
    print("Zoom search   :", ENABLE_ZOOM_SEARCH_WHEN_NO_DETECTION)
    print("Full frame too:", RUN_FULL_FRAME_DETECTION_TOO)

    # In memory-safe mode, the primary output video is written as 10% parts.
    # This avoids holding one long VideoWriter session and preserves completed work if a late frame crashes.
    writer = None
    single_writer = None
    active_chunk_info = None
    chunk_manifest_rows = []

    if SAVE_OUTPUT_VIDEO and SAVE_SINGLE_OUTPUT_VIDEO:
        single_writer = make_video_writer(OUTPUT_VIDEO_PATH, out_fps, width, height)

    preview_enabled = try_start_live_preview()

    detection_rows = []
    summary_rows = []
    class_total_counter = Counter()

    start_time = time.time()
    frame_index = 0
    current_roi = get_initial_roi(width, height)
    last_detection_frame = -1
    assignment_state = {"last_saved_frame": -10**9, "saved_count": 0, "best_conf": -1.0}

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            frame_index += 1
            if MAX_FRAMES is not None and frame_index > int(MAX_FRAMES):
                break

            # Rotate/save every 10% chunk before processing the current frame.
            if ENABLE_10_PERCENT_CHUNK_SAVE:
                new_chunk_info = get_progress_chunk_info(frame_index, total_frames)
                if active_chunk_info is None or new_chunk_info["chunk_index"] != active_chunk_info["chunk_index"]:
                    if active_chunk_info is not None:
                        if writer is not None:
                            writer.release()
                            writer = None
                        flush_chunk_rows(
                            detection_rows,
                            summary_rows,
                            active_chunk_info,
                            chunk_manifest_rows,
                            reason="10_percent_boundary",
                        )
                    active_chunk_info = dict(new_chunk_info)
                    writer, video_part = open_chunk_video_writer(active_chunk_info, out_fps, width, height)
                    active_chunk_info["video_part"] = video_part

            frame_start = time.time()
            timestamp_msec = cap.get(cv2.CAP_PROP_POS_MSEC)
            timestamp_sec = float(timestamp_msec) / 1000.0 if timestamp_msec and timestamp_msec > 0 else frame_index / out_fps

            # 1) Optional full-frame detection.
            full_frame_boxes = []
            full_raw_count = 0
            full_filtered_count = 0
            if RUN_FULL_FRAME_DETECTION_TOO:
                raw_full_boxes = predict_with_ensemble_on_frame(
                    models=models,
                    model_labels=model_labels,
                    frame_bgr=frame,
                    class_names=class_names,
                    imgsz=IMG_SIZE,
                    conf=RAW_MODEL_CONF_THRES,
                )
                full_frame_boxes, filtered_full_boxes, _ = run_postprocess_pipeline(raw_full_boxes, frame.shape)
                full_raw_count = len(raw_full_boxes)
                full_filtered_count = len(filtered_full_boxes)

            # 2) Zoom ROI inspection. This is the main assignment view.
            zoom_result = choose_zoom_inspection(
                models=models,
                model_labels=model_labels,
                class_names=class_names,
                frame=frame,
                frame_index=frame_index,
                current_roi=current_roi,
                last_detection_frame=last_detection_frame,
            )

            if zoom_result is None:
                zoom_roi = current_roi
                zoom_crop = frame.copy()
                zoom_local_boxes = []
                zoom_global_boxes = []
                zoom_raw_count = 0
                zoom_filtered_count = 0
                roi_name = "none"
                search_candidate_count = 0
            else:
                zoom_roi = zoom_result["roi"]
                current_roi = zoom_result["next_roi"]
                zoom_crop = zoom_result["crop"]
                zoom_local_boxes = zoom_result["local_boxes"]
                zoom_global_boxes = zoom_result["global_boxes"]
                zoom_raw_count = len(zoom_result["raw_local_boxes"])
                zoom_filtered_count = len(zoom_result["filtered_local_boxes"])
                roi_name = zoom_result["roi_name"]
                search_candidate_count = int(zoom_result.get("candidate_count", 1))

            if zoom_global_boxes:
                last_detection_frame = frame_index

            # 3) Combine boxes for drawing on original frame.
            # Zoom crop boxes are mapped back to original coordinates.
            all_draw_boxes = []
            all_draw_boxes.extend(full_frame_boxes)
            all_draw_boxes.extend(zoom_global_boxes)
            all_draw_boxes = nms_ensemble_boxes(all_draw_boxes, iou_thres=ENSEMBLE_NMS_IOU_THRES) if all_draw_boxes else []
            all_draw_boxes = sorted(all_draw_boxes, key=lambda x: float(x["confidence"]), reverse=True)

            for box in all_draw_boxes:
                class_total_counter[box["class_name"]] += 1

            # 4) Draw original view, ROI, zoom detections, and zoom layer.
            plotted = draw_prediction_boxes_from_list(frame, all_draw_boxes, title_prefix="PRED", box_color=(255, 255, 0))

            if DRAW_ROI_RECTANGLE_ON_ORIGINAL and ENABLE_TURBINE_ZOOM_LAYER:
                plotted = draw_roi_rectangle(plotted, zoom_roi, label="TURBINE ZOOM ROI")

            zoom_crop_drawn = draw_prediction_boxes_from_list(zoom_crop, zoom_local_boxes, title_prefix="ZOOM", box_color=(0, 255, 255))

            if ENABLE_TURBINE_ZOOM_LAYER:
                plotted = overlay_zoom_layer(
                    plotted,
                    zoom_crop_drawn,
                    zoom_roi,
                    len(zoom_local_boxes),
                    roi_name,
                )

            elapsed_sec = time.time() - start_time
            frame_time = max(time.time() - frame_start, 1e-9)
            fps_now = 1.0 / frame_time
            plotted = draw_frame_info(
                plotted,
                frame_index,
                total_frames,
                len(all_draw_boxes),
                len(zoom_local_boxes),
                fps_now,
                elapsed_sec,
                roi_name,
            )

            maybe_save_assignment_images(frame_index, plotted, zoom_crop_drawn, zoom_local_boxes, assignment_state)

            if writer is not None:
                writer.write(plotted)
            if single_writer is not None:
                single_writer.write(plotted)

            if preview_enabled:
                preview_enabled = show_live_preview_frame(plotted)
                if not preview_enabled:
                    break

            # 5) Logs.
            for box in all_draw_boxes:
                detection_rows.append({
                    "frame_index": frame_index,
                    "timestamp_sec": timestamp_sec,
                    "class_id": int(box["class_id"]),
                    "class_name": box["class_name"],
                    "confidence": float(box["confidence"]),
                    "mean_confidence": float(box.get("mean_confidence", box["confidence"])),
                    "class_threshold_used": float(box.get("class_threshold_used", get_class_threshold(box["class_name"]))),
                    "prediction_source": box.get("source", "raw"),
                    "roi_name": box.get("roi_name", roi_name),
                    "source_models": box.get("source_models", box.get("source_model", "")),
                    "ensemble_support_count": int(box.get("ensemble_support_count", 1)),
                    "ensemble_model_count": len(models),
                    "merged_from_count": int(box.get("merged_from_count", 1)),
                    "is_damage_like": bool(is_damage_class(box["class_name"])),
                    "is_contamination_like": bool(is_contamination_class(box["class_name"])),
                    "xmin": float(box["xmin"]),
                    "ymin": float(box["ymin"]),
                    "xmax": float(box["xmax"]),
                    "ymax": float(box["ymax"]),
                    "zoom_roi_xmin": float(zoom_roi[0]),
                    "zoom_roi_ymin": float(zoom_roi[1]),
                    "zoom_roi_xmax": float(zoom_roi[2]),
                    "zoom_roi_ymax": float(zoom_roi[3]),
                })

            summary_rows.append({
                "frame_index": frame_index,
                "timestamp_sec": timestamp_sec,
                "full_raw_candidate_count": full_raw_count,
                "full_filtered_candidate_count": full_filtered_count,
                "zoom_raw_candidate_count": zoom_raw_count,
                "zoom_filtered_candidate_count": zoom_filtered_count,
                "zoom_final_detection_count": len(zoom_local_boxes),
                "final_draw_detection_count": len(all_draw_boxes),
                "roi_name": roi_name,
                "search_candidate_count": search_candidate_count,
                "zoom_roi_xmin": float(zoom_roi[0]),
                "zoom_roi_ymin": float(zoom_roi[1]),
                "zoom_roi_xmax": float(zoom_roi[2]),
                "zoom_roi_ymax": float(zoom_roi[3]),
                "frame_processing_sec": frame_time,
                "instant_fps": fps_now,
            })

            if frame_index == 1 or frame_index % PRINT_EVERY_N_FRAMES == 0:
                percent = (frame_index / total_frames * 100.0) if total_frames > 0 else 0.0
                print(
                    f"Frame {frame_index}/{total_frames if total_frames > 0 else '?'} "
                    f"({percent:.1f}%) | draw={len(all_draw_boxes)} | "
                    f"zoom={len(zoom_local_boxes)} | roi={roi_name} | "
                    f"speed={fps_now:.2f} FPS | elapsed={seconds_to_hhmmss(elapsed_sec)} | "
                    f"chunk={active_chunk_info['chunk_name'] if active_chunk_info else 'none'}"
                )

            if MEMORY_CLEANUP_EVERY_N_FRAMES and frame_index % int(MEMORY_CLEANUP_EVERY_N_FRAMES) == 0:
                cleanup_runtime_memory(f"periodic frame cleanup at frame {frame_index}")

    finally:
        cap.release()
        if writer is not None:
            writer.release()
        if single_writer is not None:
            single_writer.release()
        safely_close_live_preview(preview_enabled)

    total_elapsed = time.time() - start_time
    avg_fps = frame_index / total_elapsed if total_elapsed > 0 else 0.0

    # Flush the last active 10% chunk. In memory-safe mode, detection_rows/summary_rows
    # are intentionally NOT kept until the end. They are saved per chunk and then cleared.
    if ENABLE_10_PERCENT_CHUNK_SAVE and active_chunk_info is not None:
        flush_chunk_rows(
            detection_rows,
            summary_rows,
            active_chunk_info,
            chunk_manifest_rows,
            reason="final_flush",
        )
        save_chunk_manifest(chunk_manifest_rows)
    elif SAVE_DETECTION_CSV:
        pd.DataFrame(detection_rows).to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8-sig")
        pd.DataFrame(summary_rows).to_csv(OUTPUT_SUMMARY_PATH, index=False, encoding="utf-8-sig")

    print("\n" + "=" * 70)
    print("ZOOM ROI VIDEO DETECTION COMPLETE")
    print("=" * 70)
    print("Processed frames     :", frame_index)
    print("Total elapsed        :", seconds_to_hhmmss(total_elapsed))
    print("Average FPS          :", f"{avg_fps:.3f}")
    print("Saved single video   :", OUTPUT_VIDEO_PATH if (SAVE_OUTPUT_VIDEO and SAVE_SINGLE_OUTPUT_VIDEO) else "disabled")
    print("Saved video chunks   :", CHUNK_VIDEO_DIR if (SAVE_OUTPUT_VIDEO and SAVE_CHUNK_VIDEO_PARTS) else "disabled")
    print("Saved chunk CSVs     :", CHUNK_CSV_DIR if SAVE_DETECTION_CSV else "disabled")
    print("Saved chunk manifest :", CHUNK_MANIFEST_PATH if SAVE_DETECTION_CSV else "disabled")
    print("Assignment image dir :", ASSIGNMENT_IMAGE_DIR)
    print("Assignment images    :", assignment_state.get("saved_count", 0))
    if SAVE_BEST_ASSIGNMENT_IMAGE and assignment_state.get("best_conf", -1.0) >= 0:
        print("Best assignment image:", ASSIGNMENT_IMAGE_DIR / "best_overlay_frame.jpg")
        print("Best zoom crop       :", ASSIGNMENT_IMAGE_DIR / "best_zoom_crop.jpg")

    if class_total_counter:
        print("\nDetection count by class:")
        for class_name, count in class_total_counter.most_common():
            print(f"- {class_name}: {count}")
    else:
        print("\nNo final detections were drawn.")
        print("Try one of these settings:")
        print("1) lower PRED_CONF_THRES/DAMAGE_CONF_THRES/CONTAMINATION_CONF_THRES")
        print("2) set MANUAL_TURBINE_ROI_NORM around the turbine")
        print("3) set RUN_FULL_FRAME_DETECTION_TOO = True for an extra whole-frame pass")


## 4. Run detection

This starts the full 6-model ensemble zoom-ROI video detection.  
The output is saved in chunked 10% parts to reduce memory pressure.


In [14]:
run_video_detection()


0702.MP4 ZOOM ROI ENSEMBLE VIDEO DETECTION
This script does NOT build or validate any dataset.
It reads 0702.mp4, crops/zooms the turbine region, runs the full ensemble, draws boxes, and saves the result video.

Video source: /home/jovyan/work/AIFFEL_QUEST_ENG/Main_Quest/MQ02/0703.mp4
Output video: chunked video parts
Chunk videos: /home/jovyan/work/AIFFEL_QUEST_ENG/Main_Quest/MQ02/video_0703_zoom_ensemble_result/chunks_10_percent/video_parts
Chunk CSVs  : /home/jovyan/work/AIFFEL_QUEST_ENG/Main_Quest/MQ02/video_0703_zoom_ensemble_result/chunks_10_percent/csv_parts
Assignment images: /home/jovyan/work/AIFFEL_QUEST_ENG/Main_Quest/MQ02/video_0703_zoom_ensemble_result/assignment_detection_images

LOAD ENSEMBLE MODEL(S)
CUDA available: True
Device        : 0
Half precision: True
Loading: /home/jovyan/work/AIFFEL_QUEST_ENG/Main_Quest/MQ02/yolov8s.pt
Model summary (fused): 73 layers, 11,126,358 parameters, 0 gradients, 28.4 GFLOPs
Loading: /home/jovyan/work/AIFFEL_QUEST_ENG/Main_Quest/MQ02/y

## 5. Optional: merge 10% video chunks with ffmpeg

Run this in a terminal from the chunk output folder if you want one final MP4 file:

```bash
ffmpeg -f concat -safe 0 -i video_concat_list.txt -c copy 0703_zoom_layer_ensemble_detected_full.mp4
```


# 우리가 만든 모델을 영상에 넣어보다

영상에 넣어 실행 해 보니  

아직 터빈의 오염 및 손상을 잘 탐지하는것 같지는 않지만  

잘 안보이는 부분을 스스로 확대 해서 탐지하려는 모습도 보입니다  

모델 학습을 더 공부해서 성능을 올릴수 있으면 훨씬 더 좋은 모델이 될것 같습니다  

영상 파일은 발표자 민욱 님께 전달 드렸습니다  
